### Structured Output using LangChain and Google Gemini

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

# Set API key from environment variables
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

#### 1. Define the desired output structure using Pydantic

In [6]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="The name of the person")
    age: int = Field(description="The age of the person")
    occupation: str = Field(description="The occupation or job of the person")

#### 2. Initialize the Gemini Model and extract structured data
We use `gemini-2.5-flash` or `gemini-3.5-flash` because they are verified to support structured output on your API key.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the model
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Bind structured output
structured_llm = model.with_structured_output(Person)

# Invoke the model to extract schema-compliant data
result = structured_llm.invoke("Jane Doe is a 28-year-old Software Engineer.")
result

Person(name='Jane Doe', age=28, occupation='Software Engineer')

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [9]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.3'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x111108ec0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'descr

In [10]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='**Inception** is a monumental 2010 science fiction action film written and directed by Christopher Nolan, renowned for its complex narrative, stunning visual effects, and philosophical themes. It\'s often described as a "heist movie set in the architecture of the mind."\n\nHere are the details:\n\n---\n\n### **Core Premise**\n\nThe film centers on **Dom Cobb (Leonardo DiCaprio)**, a skilled "extractor" who specializes in corporate espionage by entering people\'s dreams to steal their subconscious ideas. His unique ability has made him a fugitive, costing him everything he loves. He\'s offered a chance at redemption: to perform the inverse, an "inception" – planting an idea into a target\'s mind rather than stealing one. If successful, his criminal record will be cleared, allowing him to return home to his children.\n\n### **Plot Summary**\n\nCobb is approached by a powerful Japanese businessman, **Saito (Ken Watanabe)**, who wants an idea planted into the mind of his

In [11]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="The year the movie was released")
    direction:str=Field(..., description="The director of the movie")
    ration:float = Field(..., description="The movie ration ourt of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("provide the details about the movie spiderman : the brand new day")
response

{'raw': AIMessage(content='{\n  "title": "Spider-Man: The Brand New Day",\n  "year": 2025,\n  "direction": "Michael Green",\n  "ration": 7.8\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01e18-1050-7471-81bc-9c86e405cf4b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 1101, 'total_tokens': 1115, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1054}}),
 'parsed': Movie(title='Spider-Man: The Brand New Day', year=2025, direction='Michael Green', ration=7.8),
 'parsing_error': None}

In [13]:
response['parsed']

Movie(title='Spider-Man: The Brand New Day', year=2025, direction='Michael Green', ration=7.8)

## Nested Structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie Avengers")
response

MovieDetails(title='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Iron Man'), Actor(name='Chris Evans', role='Captain America'), Actor(name='Mark Ruffalo', role='Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Black Widow'), Actor(name='Jeremy Renner', role='Hawkeye')], genres=['Action', 'Sci-Fi', 'Adventure'], budget=220.0)

## TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [16]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie with details."""
    title:Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"the year the movie was relesed"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movies rating out of 10"]

model_withtype = model.with_structured_output(MovieDict)
response = model_withtype.invoke("Please provide the details of the movie avenger the end game")
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony and Joe Russo',
 'rating': 8.4}

In [18]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million Usd")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response



{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Sci-Fi', 'Superhero'],
 'budget': 220000000}

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

# Set API key from environment variables
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str=Field(description="The name of the person")
    email: str=Field(description="The email address of the person")
    phone: str=Field(description="The phone number of the person")


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role" : "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e946a971-c9fc-4ec8-be11-6a1acc186442'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01e42-88b9-76b2-ac48-0b3345747487-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 158, 'total_tokens': 187, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 127}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [24]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

# Set API key from environment variables
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role" : "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [26]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')